
# Clase 4 — Parámetros de generación

En la clase anterior vimos que el **prompt** guía el comportamiento del chatbot.

En esta clase vamos a modificar **cómo genera** la respuesta el modelo usando parámetros como:

- `temperature`
- `max_new_tokens`
- `top_p`
- `top_k`
- `repetition_penalty`
- `do_sample`

La idea es probar la **misma pregunta** con varias configuraciones y comparar qué cambia.

> Antes de ejecutar: activar **GPU T4** en Colab.  
> Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4.



## 1. Instalación

Usamos el mismo modelo de la clase anterior. Esta instalación puede tardar unos minutos.


In [ ]:

!pip -q install -U \
    "transformers>=4.51,<5" \
    accelerate \
    bitsandbytes \
    "pandas==2.2.2"


In [ ]:

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Activá GPU T4 antes de continuar.")


PyTorch: 2.11.0+cu128
Pandas: 2.2.2
GPU disponible: True
GPU: Tesla T4



## 2. Carga del modelo

Usamos un modelo de Hugging Face ya entrenado.

No lo estamos entrenando. Lo estamos usando para generar respuestas.

La carga en **4 bits** reduce el consumo de memoria para que pueda ejecutarse en Colab.


In [ ]:

MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"

config_cuantizacion = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Cargando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

print("Cargando modelo...")
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    quantization_config=config_cuantizacion,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

modelo.eval()
print("Modelo cargado correctamente.")


Cargando tokenizador...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Cargando modelo...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Modelo cargado correctamente.



## 3. Prompt base

Usamos un prompt simple, parecido al que venimos trabajando.

Luego cambiaremos solamente los parámetros de generación para ver qué efecto tienen.


In [ ]:

PROMPT_SISTEMA = """
Sos un asistente educativo de Procesamiento del Lenguaje Natural.
Respondé siempre en español.
Usá un lenguaje claro, ordenado y adecuado para estudiantes de nivel terciario.
No inventes información institucional concreta.
Si no tenés datos suficientes, aclaralo.
""".strip()

print(PROMPT_SISTEMA)


Sos un asistente educativo de Procesamiento del Lenguaje Natural.
Respondé siempre en español.
Usá un lenguaje claro, ordenado y adecuado para estudiantes de nivel terciario.
No inventes información institucional concreta.
Si no tenés datos suficientes, aclaralo.



## 4. Función de generación configurable

Esta función permite probar distintos valores.

### Parámetros principales

- `temperature`: creatividad.
- `max_new_tokens`: extensión máxima de la respuesta.
- `top_p`: variedad probabilística.
- `top_k`: cantidad máxima de opciones candidatas.
- `repetition_penalty`: reduce repeticiones.
- `do_sample`: activa o desactiva la generación con variedad.


In [ ]:

def responder_con_parametros(
    pregunta,
    prompt_sistema=PROMPT_SISTEMA,
    temperature=0.3,
    max_new_tokens=220,
    top_p=0.85,
    top_k=20,
    repetition_penalty=1.08,
    do_sample=True
):
    mensajes = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": pregunta}
    ]

    texto_chat = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto_chat,
        return_tensors="pt"
    ).to(modelo.device)

    parametros = {
        "max_new_tokens": int(max_new_tokens),
        "repetition_penalty": float(repetition_penalty),
        "pad_token_id": tokenizer.eos_token_id
    }

    if do_sample:
        parametros.update({
            "do_sample": True,
            "temperature": float(temperature),
            "top_p": float(top_p),
            "top_k": int(top_k)
        })
    else:
        parametros.update({
            "do_sample": False
        })

    with torch.inference_mode():
        salida = modelo.generate(
            **entradas,
            **parametros
        )

    tokens_nuevos = salida[0][entradas["input_ids"].shape[-1]:]

    respuesta = tokenizer.decode(
        tokens_nuevos,
        skip_special_tokens=True
    )

    return respuesta.strip()



## 5. Primera prueba

Probamos una pregunta con una configuración equilibrada.


In [ ]:

pregunta_base = "Explicá qué es la tokenización en PLN con un ejemplo simple."

respuesta = responder_con_parametros(
    pregunta_base,
    temperature=0.3,
    max_new_tokens=220,
    top_p=0.85,
    top_k=20,
    repetition_penalty=1.08,
    do_sample=True
)

print(respuesta)


Claro, vamos a explicar lo que es **la tokenización** en el procesamiento del lenguaje natural (PLN) con un ejemplo sencillo.

---

### 🔍 ¿Qué es la tokenización?

La **tokenización** es el primer paso que se hace cuando se quiere analizar un texto en un sistema de *Procesamiento del Lenguaje Natural*. Consiste en dividir un texto largo en **palabras o fragmentos más pequeños**, llamados *tokens*.

Estos tokens pueden ser palabras, frases, símbolos, números o incluso signos de puntuación. Es como "cortar" una oración en trozos más manejables para que el computador pueda analizarlos.

---

### 📌 Ejemplo simple

Imagina que tienes esta frase:

> "Me gusta comer pizza"

Ahora, aplicamos tokenización. Dividimos esa frase en partes más pequeñas:

➡️ ["Me", "gusta", "comer", "pizza"]

Cada palabra es un **



## 6. Comparar configuraciones

Vamos a hacer la misma pregunta con tres estilos:

1. **Estable:** más predecible y controlado.
2. **Equilibrado:** natural, pero todavía controlado.
3. **Creativo:** más variado, pero con más riesgo.


In [ ]:

CONFIGURACIONES = [
    {
        "nombre": "Estable / institucional",
        "temperature": 0.2,
        "max_new_tokens": 200,
        "top_p": 0.80,
        "top_k": 20,
        "repetition_penalty": 1.08,
        "do_sample": True
    },
    {
        "nombre": "Equilibrada",
        "temperature": 0.3,
        "max_new_tokens": 260,
        "top_p": 0.85,
        "top_k": 20,
        "repetition_penalty": 1.08,
        "do_sample": True
    },
    {
        "nombre": "Creativa",
        "temperature": 0.7,
        "max_new_tokens": 350,
        "top_p": 0.95,
        "top_k": 50,
        "repetition_penalty": 1.05,
        "do_sample": True
    },
    {
        "nombre": "Determinista",
        "temperature": 0.0,
        "max_new_tokens": 220,
        "top_p": 1.0,
        "top_k": 0,
        "repetition_penalty": 1.08,
        "do_sample": False
    }
]

CONFIGURACIONES


[{'nombre': 'Estable / institucional',
  'temperature': 0.2,
  'max_new_tokens': 200,
  'top_p': 0.8,
  'top_k': 20,
  'repetition_penalty': 1.08,
  'do_sample': True},
 {'nombre': 'Equilibrada',
  'temperature': 0.3,
  'max_new_tokens': 260,
  'top_p': 0.85,
  'top_k': 20,
  'repetition_penalty': 1.08,
  'do_sample': True},
 {'nombre': 'Creativa',
  'temperature': 0.7,
  'max_new_tokens': 350,
  'top_p': 0.95,
  'top_k': 50,
  'repetition_penalty': 1.05,
  'do_sample': True},
 {'nombre': 'Determinista',
  'temperature': 0.0,
  'max_new_tokens': 220,
  'top_p': 1.0,
  'top_k': 0,
  'repetition_penalty': 1.08,
  'do_sample': False}]

In [ ]:

def comparar_configuraciones(pregunta):
    registros = []

    for config in CONFIGURACIONES:
        print("Probando:", config["nombre"])

        respuesta = responder_con_parametros(
            pregunta=pregunta,
            temperature=config["temperature"],
            max_new_tokens=config["max_new_tokens"],
            top_p=config["top_p"],
            top_k=config["top_k"],
            repetition_penalty=config["repetition_penalty"],
            do_sample=config["do_sample"]
        )

        registros.append({
            "configuración": config["nombre"],
            "temperature": config["temperature"],
            "max_new_tokens": config["max_new_tokens"],
            "top_p": config["top_p"],
            "top_k": config["top_k"],
            "repetition_penalty": config["repetition_penalty"],
            "do_sample": config["do_sample"],
            "respuesta": respuesta
        })

    return pd.DataFrame(registros)


In [ ]:

pregunta_comparacion = "Explicá la diferencia entre un chatbot por reglas y un chatbot con IA."

df_comparacion = comparar_configuraciones(pregunta_comparacion)
df_comparacion


Probando: Estable / institucional
Probando: Equilibrada
Probando: Creativa


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Probando: Determinista


,configuración,temperature,max_new_tokens,top_p,top_k,repetition_penalty,do_sample,respuesta
0,Estable / institucional,0.2,200,0.80,20,1.08,True,¡Claro! Vamos a explicar la diferencia entre u...
1,Equilibrada,0.3,260,0.85,20,1.08,True,"Claro, vamos a explicar la diferencia entre un..."
2,Creativa,0.7,350,0.95,50,1.05,True,¡Claro! Vamos a explicar la diferencia entre u...
3,Determinista,0.0,220,1.00,0,1.08,False,"Claro, vamos a explicar la diferencia entre un..."



## 7. Ver las respuestas una por una

La tabla puede quedar incómoda si las respuestas son largas. Esta celda imprime cada resultado de forma más clara.


In [ ]:

for _, fila in df_comparacion.iterrows():
    print("=" * 90)
    print("CONFIGURACIÓN:", fila["configuración"])
    print("temperature:", fila["temperature"], "| max_new_tokens:", fila["max_new_tokens"], "| top_p:", fila["top_p"])
    print("-" * 90)
    print(fila["respuesta"])
    print()


CONFIGURACIÓN: Estable / institucional
temperature: 0.2 | max_new_tokens: 200 | top_p: 0.8
------------------------------------------------------------------------------------------
¡Claro! Vamos a explicar la diferencia entre un **chatbot por reglas** y un **chatbot con inteligencia artificial (IA)** de forma clara y sencilla, como si estuvieras hablando con un estudiante de nivel terciario.

---

### 📌 ¿Qué es un chatbot por reglas?

Un **chatbot por reglas** funciona como una máquina que sigue un conjunto de instrucciones fijas o "reglas".

#### 🔍 Cómo funciona:
- Se programan respuestas específicas para cada tipo de pregunta.
- Por ejemplo:  
  - Si el usuario dice *"¿Cuál es el horario de atención?"*, el bot responde: *"De lunes a viernes, de 9 a 18 horas."*
- Si la pregunta no está en las reglas, el bot simplemente **no responde** o dice que no sabe.

#### ✅ Ventajas

CONFIGURACIÓN: Equilibrada
temperature: 0.3 | max_new_tokens: 260 | top_p: 0.85
---------------------------------


## 8. Valores recomendados según el uso

No existe una configuración perfecta para todo.

Depende del objetivo del chatbot.


In [ ]:

recomendaciones = pd.DataFrame([
    {
        "uso": "Reglamentos o información institucional",
        "temperature": "0.2 a 0.3",
        "max_new_tokens": "200 a 350",
        "motivo": "Necesitamos respuestas estables y poco inventadas."
    },
    {
        "uso": "Orientación académica general",
        "temperature": "0.3 a 0.4",
        "max_new_tokens": "250 a 400",
        "motivo": "Permite naturalidad sin perder demasiado control."
    },
    {
        "uso": "Ideas, ejemplos o recomendaciones creativas",
        "temperature": "0.5 a 0.7",
        "max_new_tokens": "300 a 500",
        "motivo": "Permite más variedad en las respuestas."
    },
    {
        "uso": "Respuestas breves tipo ayuda rápida",
        "temperature": "0.2 a 0.3",
        "max_new_tokens": "80 a 180",
        "motivo": "Evita que la respuesta se vuelva demasiado larga."
    }
])

recomendaciones


,uso,temperature,max_new_tokens,motivo
0,Reglamentos o información institucional,0.2 a 0.3,200 a 350,Necesitamos respuestas estables y poco inventa...
1,Orientación académica general,0.3 a 0.4,250 a 400,Permite naturalidad sin perder demasiado control.
2,"Ideas, ejemplos o recomendaciones creativas",0.5 a 0.7,300 a 500,Permite más variedad en las respuestas.
3,Respuestas breves tipo ayuda rápida,0.2 a 0.3,80 a 180,Evita que la respuesta se vuelva demasiado larga.



## 9. Actividad de la clase

Elegí una pregunta relacionada con tu futuro chatbot y probala con las tres configuraciones.

Después completá la tabla con una observación.


In [ ]:

# Cambiá esta pregunta por una relacionada con tu proyecto.
pregunta_de_mi_chatbot = "¿Qué debería responder un chatbot de orientación académica si un estudiante pregunta por requisitos de inscripción?"

df_mi_prueba = comparar_configuraciones(pregunta_de_mi_chatbot)
df_mi_prueba

for indice, fila in df_mi_prueba.iterrows():
    print("=" * 90)
    print("CONFIGURACIÓN:", fila["configuración"])
    print("temperature:", fila["temperature"])
    print("max_new_tokens:", fila["max_new_tokens"])
    print("top_p:", fila["top_p"])
    print("top_k:", fila["top_k"])
    print("repetition_penalty:", fila["repetition_penalty"])
    print("do_sample:", fila["do_sample"])
    print("-" * 90)
    print("RESPUESTA COMPLETA:")
    print(fila["respuesta"])
    print()


Probando: Estable / institucional
Probando: Equilibrada
Probando: Creativa
Probando: Determinista
CONFIGURACIÓN: Estable / institucional
temperature: 0.2
max_new_tokens: 200
top_p: 0.8
top_k: 20
repetition_penalty: 1.08
do_sample: True
------------------------------------------------------------------------------------------
RESPUESTA COMPLETA:
Un chatbot de orientación académica debería responder de forma clara, precisa y accesible cuando un estudiante pregunta por los requisitos de inscripción. Aquí tienes un ejemplo de cómo podría estructurarse la respuesta:

---

**Hola, ¿en qué carrera o programa te interesas inscribir?**  
Para ayudarte mejor, necesito saber en qué área o facultad deseas inscribirte (por ejemplo: Ingeniería, Derecho, Psicología, etc.).  

Una vez que tenga esa información, te puedo dar los **requisitos específicos**, como:  
- Niveles de estudios previos (por ejemplo: secundaria completa)  
- Notas mínimas en ciertos cursos o exámenes  
- Pruebas de ingreso (si a

In [ ]:

# Completar manualmente después de leer las respuestas.
analisis = pd.DataFrame([
    {
        "configuración": "Estable / institucional",
        "claridad": "",
        "extensión": "",
        "riesgo de inventar": "",
        "¿la usaría?": "",
        "justificación": ""
    },
    {
        "configuración": "Equilibrada",
        "claridad": "",
        "extensión": "",
        "riesgo de inventar": "",
        "¿la usaría?": "",
        "justificación": ""
    },
    {
        "configuración": "Creativa",
        "claridad": "",
        "extensión": "",
        "riesgo de inventar": "",
        "¿la usaría?": "",
        "justificación": ""
    }
])

analisis



## 10. Exportar evidencia de la actividad

Esta celda guarda las respuestas en un CSV para entregar o revisar.


In [ ]:

ruta = "comparacion_parametros_clase4.csv"
df_mi_prueba.to_csv(ruta, index=False, encoding="utf-8-sig")
print("Archivo generado:", ruta)



# Cierre de la clase

Al terminar esta clase deberías poder explicar:

- qué controla `temperature`;
- qué controla `max_new_tokens`;
- para qué sirven `top_p` y `top_k`;
- qué diferencia hay entre una respuesta estable y una creativa;
- por qué un chatbot institucional necesita parámetros más conservadores;
- qué configuración inicial usarías en tu proyecto y por qué.

La próxima clase trabajaremos con **historial y memoria conversacional**.
